# Iceland Tourism - SQL Analysis
Analytical queries using DuckDB on cleaned CSV data.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.execute("CREATE VIEW overnight AS SELECT * FROM read_csv_auto('../data/processed/overnight_stays_arrivals_clean.csv')")
con.execute("CREATE VIEW occupancy AS SELECT * FROM read_csv_auto('../data/processed/occupancy_rate_clean.csv')")
con.execute("CREATE VIEW passengers AS SELECT * FROM read_csv_auto('../data/processed/keflavik_passengers_clean.csv')")
con.execute("CREATE VIEW hotels AS SELECT * FROM read_csv_auto('../data/processed/hotels_guesthouses_clean.csv')")

print("✅ Tabele załadowane!")

✅ Tabele załadowane!


In [2]:
# 1. Ranking lat wg turystów
result1 = con.execute("""
    SELECT Year, SUM(Value) as Total_Arrivals
    FROM overnight
    WHERE Citizenship = 'Foreigners'
      AND Unit = 'Arrivals'
      AND Month != 'Total'
    GROUP BY Year
    ORDER BY Total_Arrivals DESC
    LIMIT 10
""").df()

print("## Top 10 lat wg liczby turystów")
result1

## Top 10 lat wg liczby turystów


,Year,Total_Arrivals
0,2023,2716890.0
1,2024,2691701.0
2,2025,2681891.0
3,2019,2435576.0
4,2018,2342839.0
5,2022,2317940.0
6,2017,2171577.0
7,2016,1869066.0
8,2015,1337227.0
9,2021,1039802.0


In [3]:
# 2. YoY zmiana rok do roku
result2 = con.execute("""
    SELECT 
        Year,
        SUM(Value) as Total_Arrivals,
        ROUND((SUM(Value) - LAG(SUM(Value)) OVER (ORDER BY Year)) 
              / LAG(SUM(Value)) OVER (ORDER BY Year) * 100, 1) as YoY_Pct
    FROM overnight
    WHERE Citizenship = 'Foreigners'
      AND Unit = 'Arrivals'
      AND Month != 'Total'
    GROUP BY Year
    ORDER BY Year
""").df()

print("## Zmiana rok do roku")
result2

## Zmiana rok do roku


,Year,Total_Arrivals,YoY_Pct
0,1998,232260.0,NaN
1,1999,247349.0,6.5
2,2000,259279.0,4.8
3,2001,280313.0,8.1
4,2002,292262.0,4.3
5,2003,326684.0,11.8
6,2004,359974.0,10.2
7,2005,380942.0,5.8
8,2006,437561.0,14.9
9,2007,483049.0,10.4


In [4]:
# 3. Top 10 krajów Keflavik 2019
result3 = con.execute("""
    SELECT Country, SUM(Passengers) as Total_Passengers
    FROM passengers
    WHERE Year = 2019
      AND Country NOT IN ('Total Passengers', 'Foreigners', 'Iceland')
    GROUP BY Country
    ORDER BY Total_Passengers DESC
    LIMIT 10
""").df()

print("## Top 10 krajów - Keflavik 2019")
result3

## Top 10 krajów - Keflavik 2019


,Country,Total_Passengers
0,U.S.A.,464059.0
1,U.K.,261805.0
2,Other Countries,192034.0
3,Germany,132155.0
4,China,99253.0
5,France,97507.0
6,Poland,93726.0
7,Canada,69947.0
8,Spain,59141.0
9,Denmark,49280.0


In [5]:
# 4. Obłożenie wg regionu 2019
result4 = con.execute("""
    SELECT Type as Region, ROUND(AVG(Value), 1) as Avg_Occupancy
    FROM occupancy
    WHERE Year = 2019
      AND Unit = 'Occupancy rate of rooms'
      AND Type != 'Total'
    GROUP BY Type
    ORDER BY Avg_Occupancy DESC
""").df()

print("## Obłożenie hoteli wg regionu 2019")
result4

## Obłożenie hoteli wg regionu 2019


,Region,Avg_Occupancy
0,Capital region,74.8
1,Southwest,66.2
2,South,60.4
3,North,45.4
4,"West, Westfjords from 2007",43.9
5,East,42.7
